In [8]:
pip install python-dotenv


Note: you may need to restart the kernel to use updated packages.Requirement already satisfied: python-dotenv in d:\sql2019\newfolder\lib\site-packages (1.1.1)



In [1]:
# 1️⃣ Setup & Imports
import os
import cv2
import torch
import numpy as np
from PIL import Image
import whisper
import faiss
from sentence_transformers import SentenceTransformer

from transformers import BlipProcessor, BlipForConditionalGeneration
from huggingface_hub import login

from langchain_community.llms import Ollama
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain.schema import Document


d:\SQL2019\Newfolder\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [1]:
from dotenv import load_dotenv
import os

# تحميل environment variables
load_dotenv()

hf_token = os.getenv("HF_TOKEN")

if not hf_token:
    raise ValueError("❌ HF_TOKEN not found! Please set it in .env file")

print("✅ HF_TOKEN loaded:", hf_token[:10], "...")




✅ HF_TOKEN loaded: hf_ANEntlx ...


In [2]:
from huggingface_hub import login

login(token=hf_token)
print("🚀 Logged in to Hugging Face successfully")


d:\SQL2019\Newfolder\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


🚀 Logged in to Hugging Face successfully


In [3]:
# 3️⃣ Video & Audio Paths
video_path = r"C:\Users\Naira Bassam\Downloads\How Large Language Models Work.mp4"
audio_path = video_path.rsplit('.',1)[0] + ".wav"


**Extract audio from video**

In [4]:
# 4️⃣ Extract audio
os.system(f'ffmpeg -i "{video_path}" -ar 16000 -ac 1 -c:a pcm_s16le "{audio_path}" -y')
print("✅ Audio extracted:", audio_path)


✅ Audio extracted: C:\Users\Naira Bassam\Downloads\How Large Language Models Work.wav


**2:Whisper → Transcription**

In [5]:
# 5️⃣ Transcribe audio with Whisper
device = "cuda" if torch.cuda.is_available() else "cpu"
whisper_model = whisper.load_model("small", device=device)
result = whisper_model.transcribe(audio_path)
transcript = result["text"].strip()
print("✅ Transcript done:\n", transcript[:200], "...")  # first 200 chars


d:\SQL2019\Newfolder\lib\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


✅ Transcript done:
 GPT or Generative Pre-Trained Transformer is a large language model or an LLM that can generate human-like text. And I've been using GPT in its various forms for years. In this video, we are going to, ...


**3:Frames & Captions**

In [6]:
# 6️⃣ BLIP setup
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
blip_model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")
blip_model.to(device)

def generate_caption(pil_image):
    inputs = processor(pil_image, return_tensors="pt").to(device)
    with torch.no_grad():
        out = blip_model.generate(**inputs, max_length=50, num_beams=5)
    return processor.decode(out[0], skip_special_tokens=True)


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


**4:Extract frames & generate captions**

In [7]:
# 7️⃣ Extract frames every 10 seconds & generate captions
def extract_frames(video_path, interval_sec=10):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frames_info = []
    
    frame_idx = 0
    while frame_idx < total_frames:
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = cap.read()
        if not ret:
            break
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        pil_image = Image.fromarray(frame_rgb)
        caption = generate_caption(pil_image)
        timestamp_sec = frame_idx / fps
        frames_info.append({
            "frame_idx": frame_idx,
            "timestamp": timestamp_sec,
            "caption": caption
        })
        frame_idx += int(fps * interval_sec)
    cap.release()
    return frames_info

frames_info = extract_frames(video_path)
print("✅ Frames & captions done. Total frames:", len(frames_info))


✅ Frames & captions done. Total frames: 34


**5:Build FAISS Vector DB**

In [8]:
# 8️⃣ Build Vector DB
class SimpleVideoVectorDB:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)
        self.dim = self.model.get_sentence_embedding_dimension()
        self.index = faiss.IndexFlatL2(self.dim)
        self.documents, self.metadata = [], []

    def add_transcript(self, transcript):
        sentences = [s.strip() for s in transcript.replace("\n"," ").split(".") if len(s.strip())>10]
        for i, s in enumerate(sentences):
            self.documents.append(s)
            self.metadata.append({"type":"transcript","content":s,"chunk_id":f"t_{i+1}"})

    def add_frame_captions(self, frames_info):
        for f in frames_info:
            self.documents.append(f["caption"])
            self.metadata.append({"type":"visual","content":f["caption"],"timestamp":f["timestamp"]})

    def build_index(self):
        embeddings = self.model.encode(self.documents, show_progress_bar=True)
        self.index.add(embeddings.astype("float32"))

    def search(self, query, k=5):
        q_emb = self.model.encode([query])
        distances, indices = self.index.search(q_emb.astype("float32"), k)
        results = []
        for dist, idx in zip(distances[0], indices[0]):
            result = self.metadata[idx].copy()
            result["score"] = 1/(1+dist)
            results.append(result)
        return results

db = SimpleVideoVectorDB()
db.add_transcript(transcript)
db.add_frame_captions(frames_info)
db.build_index()
print("✅ Vector DB built with FAISS")


Batches: 100%|██████████| 3/3 [00:00<00:00,  4.38it/s]

✅ Vector DB built with FAISS


**6:RAG System with LangChain + Ollama**

In [9]:
# 9️⃣ Setup RAG System
rag_llm = Ollama(model="qwen3:0.6b", temperature=0.1)

prompt_template = PromptTemplate(
    input_variables=["context","question"],
    template="""You are an AI assistant. Use the context below to answer the question.

CONTEXT:
{context}

QUESTION: {question}

ANSWER:"""
)

rag_chain = LLMChain(llm=rag_llm, prompt=prompt_template)

def ask_question(query, top_k=5):
    search_results = db.search(query, k=top_k)
    context = "\n".join([f"[{r['type']}] {r['content']}" for r in search_results])
    return rag_chain.run(context=context, question=query)
# 10️⃣ Example question
question = "What is this video about?"
answer = ask_question(question)
print("\n🎯 Answer:\n", answer)


C:\Users\Naira Bassam\AppData\Local\Temp\ipykernel_7628\1468786248.py:2: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaLLM``.
  rag_llm = Ollama(model="qwen3:0.6b", temperature=0.1)
C:\Users\Naira Bassam\AppData\Local\Temp\ipykernel_7628\1468786248.py:16: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  rag_chain = LLMChain(llm=rag_llm, prompt=prompt_template)
C:\Users\Naira Bassam\AppData\Local\Temp\ipykernel_7628\1468786248.py:21: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  r


🎯 Answer:
 <think>
Okay, the user is asking what this video is about, and I need to use the context provided. Let me look at the transcript again.

The first part says, "In this video, we are going to, number one, ask what is an LLM." Then it goes on to describe how they work and lastly, think of it as training. Then there's a mention of an idea. The user's question is "What is this video about?" and the answer should be based on the context.

So the video is structured to first ask what an LLM is, then describe its working, and finally think of it as training. The answer should be that the video is about explaining the concept of an LLM. I need to make sure I'm not missing any other parts. The answer is straightforward based on the transcript. No need to add anything else.
</think>

The video is about explaining what an LLM (Large Language Model) is, how it works, and its role in training.


In [11]:
!pip freeze > requirements.txt



In [16]:
import os

# عرض الملفات
files = os.listdir()
print(files)




['blue[1].svg', 'RAG Video.ipynb', 'requirements.txt', 'session2[1].ipynb', 'session[1].ipynb', 'Untitled-1.ipynb']


In [17]:
import os

# اعرض المسار الحالي
print(os.getcwd())


c:\Users\Naira Bassam\AppData\Local\Microsoft\Windows\INetCache\IE\SZCFGYZW


In [18]:
with open("requirements.txt", "w") as f:
    f.write("""
streamlit
torch
transformers
sentence-transformers
faiss-cpu
opencv-python
Pillow
scenedetect
langchain
langchain-ollama
""")


In [20]:
import os
print(os.getcwd())
#!/usr/bin/env python3  



c:\Users\Naira Bassam\AppData\Local\Microsoft\Windows\INetCache\IE\SZCFGYZW
